In [1]:
### run this in the cellpose4 environment
import numpy as np
print(np.__file__)

C:\Users\Scope7a1\anaconda3\envs\cellpose4\Lib\site-packages\numpy\__init__.py


In [1]:
from ioMicroAllen import read_im,full_deconv
import glob,os,sys,numpy as np

In [7]:
fls = glob.glob(r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H1_NC_set*\*.zarr')


In [8]:
icol=1
from tqdm import tqdm
immed = np.median([np.array(read_im(fl)[icol][20],dtype=np.float32) for fl in tqdm(fls[:500])],axis=0)
np.savez(rf'C:\Scripts\Jess\MYH67_DNA_RNA\flat_field_kiwi\Myh67kiwiDNA_med_col_raw{icol}.npz',im=immed)

  0%|          | 0/500 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [9]:
im.shape

(2, 54, 2048, 2048)

In [7]:
#python -m pip install "cellpose[gui]"
import napari
V = napari.Viewer()
V.add_image(immed)

<Image layer 'immed' at 0x24dda904320>

In [2]:

psf_file = r'C:\Scripts\Jess\MYH67_DNA_RNA\psf_647_Kiwi.npy' 
immed = np.load(rf'C:\Scripts\Jess\MYH67_DNA_RNA\flat_field_kiwi\Myh67kiwiDNA_med_col_raw1.npz')['im']
import cv2
immed = cv2.blur(immed,(20,20))
immed = immed/np.median(immed)
psf = np.load(psf_file)

In [3]:
all_raw_fls = glob.glob(r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H1_NC_set12\*.zarr')
len(all_raw_fls)

113

In [11]:
all_raw_fls[0]

'\\\\merfish13\\Public\\Fugui_Jess\\Myh67_mouse_Kiwi_09_21_2026\\DNA\\H1_NC_set2\\Conv_zscan__087.zarr'

In [4]:
import os

segm_path = r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\segmentation'
os.makedirs(segm_path,exist_ok=True)
from tqdm import tqdm
fl_raw = all_raw_fls[76]
fov = os.path.basename(fl_raw).replace('.zarr','')
htag = os.path.basename(os.path.dirname(fl_raw))
fl_dapi = rf'{segm_path}\{fov}_{htag}.tiff'
fl_segm = rf'{segm_path}\{fov}_{htag}.npz'
if not os.path.exists(fl_segm):
    imdapi = np.array(read_im(fl_raw)[-1],dtype=np.float32)
    imdapi = imdapi/immed
    imdapid=full_deconv(imdapi,s_=400,pad=100,psf=psf,
                parameters={'method': 'wiener', 'beta': 0.005, 'niter': 50},gpu=True,force=True)

In [5]:
vmax=10000
imdapid = np.clip(imdapid,0,vmax)
imdapid_ = imdapid[::2,::3,::3][1:-1]
imdapid_ = ((imdapid_/vmax)*255).astype(np.uint8)
zplanes = np.arange(len(imdapid))[::4]#[1:-1]

import tifffile
tifffile.imwrite(
    fl_dapi,
    imdapid_,
    bigtiff=True,
    compression="zstd",
    compressionargs={"level": 5},
    photometric="minisblack",
    metadata={"axes": "ZYX"},
)

import numpy as np
from cellpose import models

model = models.CellposeModel(
    gpu=True,
    pretrained_model="cpsam_v2",
)

masks, flows, styles = model.eval(
    imdapid_.astype(np.float32),
    z_axis=0,
    channel_axis=None,

    # Segment each XY plane independently
    do_3D=False,

    # Then connect masks between adjacent planes
    stitch_threshold=0.5,

    diameter=None,
    cellprob_threshold=0.0,
    flow_threshold=0.4,
    min_size=100,

    # Normalize each channel across the whole stack
    normalize={
        "normalize": True,
        "percentile": [1, 99],
        "norm3D": True,
    },
)

C:\Users\Scope7a1\anaconda3\envs\cellpose4\Lib\site-packages\cellpose\dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),
100%|██████████| 24/24 [00:00<00:00, 55.51it/s]


In [27]:
import napari
V = napari.Viewer()
#V.add_image(imdapid)
V.add_image(imdapid_)
V.add_labels(masks)

<Labels layer 'masks' at 0x2cb34e9fda0>

In [28]:
fl_raw = all_raw_fls[40]
imdapi = read_im(fl_raw)[-1]

import napari
V = napari.Viewer()
V.add_image(imdapi)

<Image layer 'imdapi' at 0x2cb3e62ef30>

In [10]:
import os
segm_path = r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\segmentation'
os.makedirs(segm_path,exist_ok=True)
from tqdm import tqdm
for fl_raw in tqdm(all_raw_fls):
    fov = os.path.basename(fl_raw).replace('.zarr','')
    htag = os.path.basename(os.path.dirname(fl_raw))
    fl_dapi = rf'{segm_path}\{fov}_{htag}.tiff'
    fl_segm = rf'{segm_path}\{fov}_{htag}.npz'
    if not os.path.exists(fl_segm):
        imdapi = np.array(read_im(fl_raw)[-1],dtype=np.float32)
        imdapi = imdapi/immed
        imdapid=full_deconv(imdapi,s_=400,pad=100,psf=psf,
                    parameters={'method': 'wiener', 'beta': 0.005, 'niter': 50},gpu=True,force=True)
        vmax=10000
        imdapid = np.clip(imdapid,0,vmax)
        imdapid_ = imdapid[::2,::3,::3]
        imdapid_ = ((imdapid_/vmax)*255).astype(np.uint8)
        zplanes = np.arange(len(imdapid))[::2]#[1:-1]

        import tifffile
        tifffile.imwrite(
            fl_dapi,
            imdapid_,
            bigtiff=True,
            compression="zstd",
            compressionargs={"level": 5},
            photometric="minisblack",
            metadata={"axes": "ZYX"},
        )

        import numpy as np
        from cellpose import models

        model = models.CellposeModel(
            gpu=True,
            pretrained_model="cpsam_v2",
        )

        masks, flows, styles = model.eval(
            imdapid_.astype(np.float32),
            z_axis=0,
            channel_axis=None,

            # Segment each XY plane independently
            do_3D=False,

            # Then connect masks between adjacent planes
            stitch_threshold=0.5,

            diameter=None,
            cellprob_threshold=0.0,
            flow_threshold=0.4,
            min_size=100,

            # Normalize each channel across the whole stack
            normalize={
                "normalize": True,
                "percentile": [1, 99],
                "norm3D": True,
            },
        )
        np.savez_compressed(fl_segm,segm = masks,zplanes=zplanes,shape=imdapi.shape)
        if False:
            import napari
            V = napari.Viewer()
            V.add_image(imdapid_,colormap='bop blue',blending='Additive')
            V.add_labels(masks)

100%|██████████| 113/113 [09:52<00:00,  5.25s/it]
